In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pennylane as qml
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from pathlib import Path
import random
import warnings
warnings.filterwarnings('ignore')
import os
import matplotlib.pyplot as plt
import seaborn as sns

# --- QNN Model Definition ---
class QNN(nn.Module):
    def __init__(self, n_qubits, n_layers, n_classes=2):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.dev = qml.device("lightning.qubit", wires=n_qubits)

        @qml.qnode(self.dev, interface="torch", diff_method="parameter-shift")
        def circuit(inputs, weights):
            for i in range(n_qubits):
                qml.RY(inputs[i], wires=i)
                qml.RZ(inputs[i], wires=i)
            for l in range(n_layers):
                for i in range(n_qubits):
                    qml.RY(weights[l, i, 0], wires=i)
                    qml.RZ(weights[l, i, 1], wires=i)
                    qml.RX(weights[l, i, 2], wires=i)
                for i in range(n_qubits - 1):
                    qml.CNOT(wires=[i, i + 1])
                if n_qubits > 1:
                    qml.CNOT(wires=[n_qubits - 1, 0])
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

        self.circuit = circuit
        weight_shape = (n_layers, n_qubits, 3)
        self.weights = nn.Parameter(0.01 * torch.randn(weight_shape))
        self.classifier = nn.Sequential(
            nn.Linear(n_qubits, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 16), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(16, n_classes)
        )

    def forward(self, x):
        batch_size = x.shape[0]
        q_outputs = [self.circuit(x[i], self.weights) for i in range(batch_size)]
        quantum_features = torch.stack([torch.stack(out) for out in q_outputs])
        return self.classifier(quantum_features)

# --- Enhanced Data Loader ---
class OptimizedRanSAPDataLoader:
    def __init__(self, data_dir: str):
        self.data_dir = Path(data_dir)
        self.scaler = StandardScaler()

    def quick_directory_scan(self) -> list[Path]:
        print(f"Scanning directory: {self.data_dir} for all CSV files...")
        if not self.data_dir.exists():
            raise FileNotFoundError(f"Directory {self.data_dir} not found")
        csv_files = list(self.data_dir.rglob("*.csv"))
        print(f"Total CSV files found: {len(csv_files)}")
        return csv_files

    def classify_file_by_path(self, file_path: Path) -> str:
        path_str = str(file_path).lower()
        ransomware_names = ['cerber', 'darkside', 'gandcrab', 'ryuk', 'sodinokibi', 'teslacrypt', 'wannacry']
        if any(name in path_str for name in ransomware_names):
            return 'ransomware'
        return 'benign'

    def extract_quick_features(self, df: pd.DataFrame) -> list[float]:
        """Extract features with better error handling"""
        if df.empty: 
            return None
        
        try:
            features = []
            
            # Try to get numeric columns
            numeric_cols = df.select_dtypes(include=[np.number]).columns
            
            # If no numeric columns, try to convert some columns to numeric
            if len(numeric_cols) == 0:
                for col in df.columns[:5]:
                    try:
                        df[col] = pd.to_numeric(df[col], errors='coerce')
                    except:
                        continue
                numeric_cols = df.select_dtypes(include=[np.number]).columns
            
            if len(numeric_cols) == 0:
                return None
            
            # Extract statistical features from numeric columns
            for i, col in enumerate(numeric_cols):
                if i >= 3:  # Limit to first 3 numeric columns
                    break
                data = df[col].dropna()
                if len(data) > 0:
                    features.extend([
                        float(data.mean()),
                        float(data.std()) if len(data) > 1 else 0.0
                    ])
                else:
                    features.extend([0.0, 0.0])
            
            # Add shape features
            features.extend([float(len(df)), float(df.shape[1])])
            
            # Pad or trim to exactly 8 features
            while len(features) < 8: 
                features.append(0.0)
            
            return features[:8]
        except Exception as e:
            return None

    def load_balanced_sample_data(self, max_files_per_class: int = 200, max_attempts: int = 1000, 
                                 holdout_ratio: float = 0.2) -> tuple:
        """Load data with holdout set for final evaluation"""
        print("\n" + "="*50)
        print("ENHANCED DATA LOADING WITH HOLDOUT SET")
        print("="*50)

        all_csv_files = self.quick_directory_scan()
        if not all_csv_files: 
            raise ValueError("No CSV files found!")

        # Separate files into ransomware and benign lists
        ransomware_files = [f for f in all_csv_files if self.classify_file_by_path(f) == 'ransomware']
        benign_files = [f for f in all_csv_files if self.classify_file_by_path(f) == 'benign']

        print(f"Found {len(ransomware_files)} ransomware files and {len(benign_files)} benign files.")

        # Shuffle lists
        random.shuffle(ransomware_files)
        random.shuffle(benign_files)

        # Process files until we get enough valid samples
        ransomware_data = []
        benign_data = []
        
        print(f"\nTarget: {max_files_per_class} valid samples per class")
        print(f"Processing files (this may take a few minutes)...\n")
        
        # Process ransomware files
        for i, csv_file in enumerate(ransomware_files[:max_attempts]):
            if len(ransomware_data) >= max_files_per_class:
                break
            
            if (i + 1) % 100 == 0:
                print(f"  Ransomware progress: {len(ransomware_data)}/{max_files_per_class} samples (checked {i+1} files)")
            
            try:
                if csv_file.stat().st_size > 10_000_000:  # Skip files > 10MB
                    continue
                    
                for encoding in ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']:
                    try:
                        df = pd.read_csv(csv_file, nrows=2000, low_memory=False, 
                                       encoding=encoding, on_bad_lines='skip')
                        break
                    except:
                        continue
                else:
                    continue
                
                if df.empty: 
                    continue
                    
                features = self.extract_quick_features(df)
                if features is not None:
                    ransomware_data.append((features, 1, str(csv_file)))
                        
            except Exception:
                continue
        
        print(f"  Ransomware: {len(ransomware_data)} valid samples extracted")
        
        # Process benign files
        for i, csv_file in enumerate(benign_files[:max_attempts]):
            if len(benign_data) >= max_files_per_class:
                break
            
            if (i + 1) % 100 == 0:
                print(f"  Benign progress: {len(benign_data)}/{max_files_per_class} samples (checked {i+1} files)")
            
            try:
                if csv_file.stat().st_size > 10_000_000:  # Skip files > 10MB
                    continue
                    
                for encoding in ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']:
                    try:
                        df = pd.read_csv(csv_file, nrows=2000, low_memory=False, 
                                       encoding=encoding, on_bad_lines='skip')
                        break
                    except:
                        continue
                else:
                    continue
                
                if df.empty: 
                    continue
                    
                features = self.extract_quick_features(df)
                if features is not None:
                    benign_data.append((features, 0, str(csv_file)))
                        
            except Exception:
                continue
        
        print(f"  Benign: {len(benign_data)} valid samples extracted")
        
        # Balance the classes
        min_samples = min(len(ransomware_data), len(benign_data))
        if min_samples < 20:
            print(f"\nWARNING: Only {min_samples} samples per class available!")
        
        ransomware_data = ransomware_data[:min_samples]
        benign_data = benign_data[:min_samples]
        
        # Combine all data
        all_data = ransomware_data + benign_data
        random.shuffle(all_data)
        
        # Split into training/validation set and holdout test set
        holdout_size = int(len(all_data) * holdout_ratio)
        holdout_data = all_data[:holdout_size]
        train_val_data = all_data[holdout_size:]
        
        # Extract features and labels
        X_train_val = np.array([d[0] for d in train_val_data])
        y_train_val = np.array([d[1] for d in train_val_data])
        X_holdout = np.array([d[0] for d in holdout_data])
        y_holdout = np.array([d[1] for d in holdout_data])
        
        print("\n" + "="*50)
        print("DATASET SUMMARY")
        print("="*50)
        print(f"Training/Validation set: {len(X_train_val)} samples")
        print(f"  - Ransomware: {np.sum(y_train_val == 1)}")
        print(f"  - Benign: {np.sum(y_train_val == 0)}")
        print(f"Holdout test set: {len(X_holdout)} samples")
        print(f"  - Ransomware: {np.sum(y_holdout == 1)}")
        print(f"  - Benign: {np.sum(y_holdout == 0)}")
        
        return X_train_val, y_train_val, X_holdout, y_holdout

def evaluate_model(model, X, y, batch_size=8):
    """Evaluate model and return predictions and accuracy"""
    model.eval()
    dataset = TensorDataset(torch.FloatTensor(X), torch.LongTensor(y))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            outputs = model(X_batch)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    
    return np.array(all_preds), np.array(all_labels)

def train_single_fold(X_train, y_train, X_val, y_val, n_qubits=8, n_layers=3, 
                     epochs=50, batch_size=8, lr=0.001, verbose=True):
    """Train a single model and return it with training history"""
    # Prepare data
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_train_quantum = np.tanh(X_train_scaled) * np.pi
    X_val_quantum = np.tanh(X_val_scaled) * np.pi
    
    # Create data loaders
    train_dataset = TensorDataset(torch.FloatTensor(X_train_quantum), torch.LongTensor(y_train))
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    # Calculate class weights
    n_samples_0 = np.sum(y_train == 0)
    n_samples_1 = np.sum(y_train == 1)
    if n_samples_0 > 0 and n_samples_1 > 0:
        weight_for_0 = (n_samples_0 + n_samples_1) / (2.0 * n_samples_0)
        weight_for_1 = (n_samples_0 + n_samples_1) / (2.0 * n_samples_1)
        class_weights = torch.tensor([weight_for_0, weight_for_1], dtype=torch.float)
    else:
        class_weights = None
    
    # Initialize model
    model = QNN(n_qubits=n_qubits, n_layers=n_layers, n_classes=2)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    
    # Training history
    history = {'train_loss': [], 'train_acc': [], 'val_acc': []}
    
    # Training loop
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        train_preds, train_labels = [], []
        
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            
            _, predicted = torch.max(outputs, 1)
            train_preds.extend(predicted.cpu().numpy())
            train_labels.extend(y_batch.cpu().numpy())
        
        # Calculate accuracies
        train_acc = accuracy_score(train_labels, train_preds)
        val_preds, val_labels = evaluate_model(model, X_val_quantum, y_val, batch_size)
        val_acc = accuracy_score(val_labels, val_preds)
        
        # Store history
        avg_loss = total_loss / len(train_loader)
        history['train_loss'].append(avg_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        # Print progress every 5 epochs
        if verbose and (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1}/{epochs}: Loss={avg_loss:.4f}, "
                  f"Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}")
    
    return model, scaler, history

def cross_validate_qnn(X, y, n_folds=5, n_qubits=8, n_layers=3, epochs=50, batch_size=8, lr=0.001):
    """Perform k-fold cross-validation"""
    print("\n" + "="*50)
    print(f"STARTING {n_folds}-FOLD CROSS-VALIDATION")
    print("="*50)
    
    kfold = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    fold_results = []
    all_histories = []
    
    for fold, (train_idx, val_idx) in enumerate(kfold.split(X, y), 1):
        print(f"\nFold {fold}/{n_folds}:")
        print(f"  Training samples: {len(train_idx)}, Validation samples: {len(val_idx)}")
        
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        # Train model for this fold
        model, scaler, history = train_single_fold(
            X_train, y_train, X_val, y_val,
            n_qubits=n_qubits, n_layers=n_layers,
            epochs=epochs, batch_size=batch_size, lr=lr
        )
        
        # Final evaluation on validation set
        X_val_scaled = scaler.transform(X_val)
        X_val_quantum = np.tanh(X_val_scaled) * np.pi
        val_preds, val_labels = evaluate_model(model, X_val_quantum, y_val, batch_size)
        val_acc = accuracy_score(val_labels, val_preds)
        
        fold_results.append({
            'fold': fold,
            'val_accuracy': val_acc,
            'val_preds': val_preds,
            'val_labels': val_labels,
            'model': model,
            'scaler': scaler
        })
        all_histories.append(history)
        
        print(f"  Fold {fold} Validation Accuracy: {val_acc:.4f}")
    
    # Calculate average performance
    avg_accuracy = np.mean([r['val_accuracy'] for r in fold_results])
    std_accuracy = np.std([r['val_accuracy'] for r in fold_results])
    
    print("\n" + "="*50)
    print("CROSS-VALIDATION RESULTS")
    print("="*50)
    print(f"Average Validation Accuracy: {avg_accuracy:.4f} ± {std_accuracy:.4f}")
    fold_accs = [f'{r["val_accuracy"]:.4f}' for r in fold_results]
    print(f"Individual Fold Accuracies: {fold_accs}")
    
    return fold_results, all_histories

def evaluate_on_holdout(fold_results, X_holdout, y_holdout, batch_size=8):
    """Evaluate best model from CV on holdout test set"""
    print("\n" + "="*50)
    print("FINAL EVALUATION ON HOLDOUT TEST SET")
    print("="*50)
    
    # Select best model from CV
    best_fold = max(fold_results, key=lambda x: x['val_accuracy'])
    best_model = best_fold['model']
    best_scaler = best_fold['scaler']
    
    print(f"Using best model from Fold {best_fold['fold']} "
          f"(Val Accuracy: {best_fold['val_accuracy']:.4f})")
    
    # Prepare holdout data
    X_holdout_scaled = best_scaler.transform(X_holdout)
    X_holdout_quantum = np.tanh(X_holdout_scaled) * np.pi
    
    # Evaluate on holdout set
    holdout_preds, holdout_labels = evaluate_model(best_model, X_holdout_quantum, y_holdout, batch_size)
    holdout_acc = accuracy_score(holdout_labels, holdout_preds)
    
    print(f"\nHoldout Test Set Accuracy: {holdout_acc:.4f}")
    print("\nDetailed Classification Report:")
    print(classification_report(holdout_labels, holdout_preds, 
                               target_names=['Benign', 'Ransomware'], 
                               digits=4))
    
    # Confusion Matrix
    cm = confusion_matrix(holdout_labels, holdout_preds)
    print("\nConfusion Matrix:")
    print(f"              Predicted")
    print(f"              Benign  Ransomware")
    print(f"Actual Benign     {cm[0,0]:3d}     {cm[0,1]:3d}")
    print(f"     Ransomware   {cm[1,0]:3d}     {cm[1,1]:3d}")
    
    return holdout_acc, holdout_preds, holdout_labels

def plot_training_history(all_histories):
    """Plot training curves from all folds"""
    try:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        
        # Plot loss curves
        for i, history in enumerate(all_histories):
            epochs = range(1, len(history['train_loss']) + 1)
            axes[0].plot(epochs, history['train_loss'], alpha=0.7, label=f'Fold {i+1}')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].set_title('Training Loss Across Folds')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Plot training accuracy curves
        for i, history in enumerate(all_histories):
            epochs = range(1, len(history['train_acc']) + 1)
            axes[1].plot(epochs, history['train_acc'], alpha=0.7, label=f'Fold {i+1}')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Accuracy')
        axes[1].set_title('Training Accuracy Across Folds')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        # Plot validation accuracy curves
        for i, history in enumerate(all_histories):
            epochs = range(1, len(history['val_acc']) + 1)
            axes[2].plot(epochs, history['val_acc'], alpha=0.7, label=f'Fold {i+1}')
        axes[2].set_xlabel('Epoch')
        axes[2].set_ylabel('Accuracy')
        axes[2].set_title('Validation Accuracy Across Folds')
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig('qnn_training_history.png', dpi=100, bbox_inches='tight')
        plt.show()
        print("\nTraining history plots saved as 'qnn_training_history.png'")
    except Exception as e:
        print(f"Could not create plots: {e}")

def main_training_pipeline(data_dir: str, target_samples_per_class: int = 200, 
                          n_folds: int = 5, n_qubits: int = 8, n_layers: int = 3,
                          epochs: int = 50, batch_size: int = 8, lr: float = 0.001):
    """Main training pipeline with cross-validation and holdout evaluation"""
    print("\n" + "="*60)
    print("QUANTUM NEURAL NETWORK - ENHANCED TRAINING PIPELINE")
    print("="*60)
    print(f"Configuration:")
    print(f"  - Target samples per class: {target_samples_per_class}")
    print(f"  - Cross-validation folds: {n_folds}")
    print(f"  - Quantum circuit: {n_qubits} qubits, {n_layers} layers")
    print(f"  - Training: {epochs} epochs, batch size {batch_size}, lr {lr}")
    
    # Load data
    data_loader = OptimizedRanSAPDataLoader(data_dir)
    try:
        X_train_val, y_train_val, X_holdout, y_holdout = data_loader.load_balanced_sample_data(
            max_files_per_class=target_samples_per_class,
            max_attempts=2000,  # Increased to ensure we get enough samples
            holdout_ratio=0.2
        )
        
        # Check if we have enough data
        if len(X_train_val) < 20:
            print(f"\nERROR: Only {len(X_train_val)} training samples available.")
            print("Cannot proceed with cross-validation. Need at least 20 samples.")
            return None
        
        # Perform cross-validation
        fold_results, all_histories = cross_validate_qnn(
            X_train_val, y_train_val,
            n_folds=n_folds, n_qubits=n_qubits, n_layers=n_layers,
            epochs=epochs, batch_size=batch_size, lr=lr
        )
        
        # Evaluate on holdout set
        if len(X_holdout) > 0:
            holdout_acc, holdout_preds, holdout_labels = evaluate_on_holdout(
                fold_results, X_holdout, y_holdout, batch_size
            )
        else:
            print("\nNo holdout set available for final evaluation.")
        
        # Plot training history
        plot_training_history(all_histories)
        
        # Return best model
        best_fold = max(fold_results, key=lambda x: x['val_accuracy'])
        return best_fold['model']
        
    except Exception as e:
        print(f"\nError during training pipeline: {e}")
        import traceback
        traceback.print_exc()
        return None

# --- Main Execution ---
if __name__ == "__main__":
    # !! IMPORTANT !!
    # Update this path to your RanSAP dataset location
    data_dir = "/kaggle/input/ransap-2022-ransomware-behavioral-features/RanSAP/dataset"
    
    if not os.path.exists(data_dir):
        print(f"FATAL: Path does not exist -> '{data_dir}'")
        print("Please update the data_dir variable with the correct path to your RanSAP dataset.")
    else:
        # Run the enhanced training pipeline
        model = main_training_pipeline(
            data_dir=data_dir,
            target_samples_per_class=200,  # Increased from 50 to 200
            n_folds=5,                      # 5-fold cross-validation
            n_qubits=8,
            n_layers=3,
            epochs=50,                      # Increased epochs
            batch_size=8,
            lr=0.001
        )


QUANTUM NEURAL NETWORK - ENHANCED TRAINING PIPELINE
Configuration:
  - Target samples per class: 200
  - Cross-validation folds: 5
  - Quantum circuit: 8 qubits, 3 layers
  - Training: 50 epochs, batch size 8, lr 0.001

ENHANCED DATA LOADING WITH HOLDOUT SET
Scanning directory: /kaggle/input/ransap-2022-ransomware-behavioral-features/RanSAP/dataset for all CSV files...
Total CSV files found: 2990
Found 2388 ransomware files and 602 benign files.

Target: 200 valid samples per class
Processing files (this may take a few minutes)...

  Ransomware progress: 23/200 samples (checked 100 files)
  Ransomware progress: 39/200 samples (checked 200 files)
  Ransomware progress: 56/200 samples (checked 300 files)
  Ransomware progress: 75/200 samples (checked 400 files)
  Ransomware progress: 98/200 samples (checked 500 files)
  Ransomware progress: 118/200 samples (checked 600 files)
  Ransomware progress: 131/200 samples (checked 700 files)
  Ransomware progress: 147/200 samples (checked 800 f